# 🧠 HollyShit Phase 5: 双脑架构 Stage 1 (单流右脑 + 上下文采择器)

- **主干流**: 完整问题前缀 + 回答 / 长文滚动续写 (T=32 子句，混合比 30%)
- **动态探索度**: 32-Band 自适应先验 (σ_sem / σ_syn) + 4-Way 软竞争流匹配 (65/35)
- **独立采择器**: 上下文候选池罕见跨度召回 + “不抄 (Sink)” 空门控
- **算子说明**: 纯 PyTorch 高效实现 (子句级 T=32 耗时 <10ms)，无需本地编译 C++/CUDA Mamba

In [ ]:
# 0. 清理旧版 Checkpoint (如需全新训练可取消注释并运行本单元格，否则保留以支持断点续训)
# !rm -rf /content/drive/MyDrive/hollyshit/checkpoints/run/right_brain_stage1_step_*
# !rm -rf /content/drive/MyDrive/hollyshit/checkpoints/run/selector_ctx_step_*


In [ ]:
from google.colab import drive
import os

# 1. 挂载 Google Drive
drive.mount('/content/drive')

# 2. 安装必要依赖 (纯 PyTorch 实现，秒级就绪)
!pip install -q pyarrow datasets

print("
" + "="*50)
print("✅ 工作区初始化完成！当前目录:", os.getcwd())
print("="*50)


In [ ]:
# 2. 从 GitHub 拉取纯代码 (如果已存在则 git pull)
if not os.path.exists('/content/HollyShit'):
    !git clone https://github.com/catclever/HollyShit.git /content/HollyShit
    %cd /content/HollyShit
    !git submodule update --init training/core
else:
    %cd /content/HollyShit
    !git pull
    !git submodule update --init training/core

# 3. 0秒软链接：把云盘里的大权重和数据集挂入项目（物理存储仍在云盘，0重复占用）
!ln -sfn /content/drive/MyDrive/hollyshit/checkpoints checkpoints
!ln -sfn /content/drive/MyDrive/hollyshit/data data


In [ ]:
import os

new_data = False
drive_corpus_path = "/content/drive/MyDrive/hollyshit/data/omni_knowledge_corpus.jsonl"
local_corpus_path = "/content/omni_knowledge_corpus.jsonl"

if not new_data and os.path.exists(drive_corpus_path):
    if not os.path.exists(local_corpus_path):
        print(f"📦 正在从 Google Drive 拷贝语料至本地 SSD (加速训练 IO)...")
        !cp /content/drive/MyDrive/hollyshit/data/omni_knowledge_corpus.jsonl /content/omni_knowledge_corpus.jsonl
    print(f"✅ 已加载语料: {local_corpus_path}")
else:
    !python scripts/build_omni_knowledge_corpus.py --output /content/omni_knowledge_corpus.jsonl


In [ ]:
# ==============================================================================
# 🚀 Phase 5 · Stage 1 单流右脑训练
# 主干流 = 问答完整前缀 + 回答 / 长文滚动续写 (T=32，混合比例 30%)
# ==============================================================================
!python -u training/train_phase5_right_brain_stage1.py \n  --qa_paths /content/omni_knowledge_corpus.jsonl \n  --cont_paths /content/omni_knowledge_corpus.jsonl \n  --cont_ratio 0.3 \n  --max_clauses 32 \n  --batch_size 32 \n  --accum_steps 2 \n  --d_mamba 1024 \n  --epochs 3 \n  --lr 3e-4 \n  --mixed_precision bf16 \n  --out_dir /content/drive/MyDrive/hollyshit/checkpoints/run \n  --encoder_ckpt /content/drive/MyDrive/hollyshit/checkpoints/basencoder_step_115000/student.pt \n  --left_brain_ckpt_prefix left_brain_s32_fullrank \n  --eval_interval 200 \n  --save_steps 1000 \n  --auto_resume


In [ ]:
# ==============================================================================
# 🎯 Phase 5 · 上下文知识采择器训练 (Context KnowledgeSelector)
# 独立学习从上文候选池中召回罕见跨度相关子句，具备“不抄 (Sink)”空门控
# ==============================================================================
!python -u training/train_phase5_selector.py \n  --qa_paths /content/omni_knowledge_corpus.jsonl \n  --cont_paths /content/omni_knowledge_corpus.jsonl \n  --cont_ratio 0.3 \n  --max_clauses 32 \n  --batch_size 32 \n  --accum_steps 1 \n  --epochs 3 \n  --lr 3e-4 \n  --mixed_precision bf16 \n  --out_dir /content/drive/MyDrive/hollyshit/checkpoints/run \n  --encoder_ckpt /content/drive/MyDrive/hollyshit/checkpoints/basencoder_step_115000/student.pt \n  --left_brain_ckpt_prefix left_brain_s32_fullrank \n  --eval_interval 200 \n  --save_steps 1000 \n  --auto_resume


In [ ]:
import os
import re
import glob
import time
from google.colab import drive, runtime

drive_ckpt_dir = "/content/drive/MyDrive/hollyshit/checkpoints/run"

def _extract_step(path_str: str) -> int:
    m = re.search(r"step_(\d+)", os.path.basename(path_str))
    return int(m.group(1)) if m else -1

rb_ckpts = sorted(glob.glob(f"{drive_ckpt_dir}/right_brain_stage1_step_*"), key=_extract_step)
sel_ckpts = sorted(glob.glob(f"{drive_ckpt_dir}/selector_ctx_step_*"), key=_extract_step)

print("="*60)
if rb_ckpts:
    print(f"🏆 最新右脑检查点: {os.path.basename(rb_ckpts[-1])}")
if sel_ckpts:
    print(f"🎯 最新采择器检查点: {os.path.basename(sel_ckpts[-1])}")
print("="*60)

# 强制刷写 Google Drive FUSE 写入缓存
print("🔄 正在强制刷写 Google Drive 写入缓存...")
try:
    drive.flush_and_unmount()
    print("✅ Google Drive 缓存已完全刷写完毕！")
except Exception as e:
    print(f"⚠️ 缓存刷写提示: {e}")

# 可选：训练完成后自动下线释放 GPU 运行时 (节省算力额度)
# time.sleep(5)
# runtime.unassign()
